In [1]:
# === CELL 1: SETUP & CONFIGURATION ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm  # For MobileNetV3 Backbone
import numpy as np
import os
import cv2
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast # Crucial for 3080 Ti speed

# Hardware Check
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE} (AMP Enabled)")

# GLOBAL CONFIGURATION
CONFIG = {
    # Input
    'img_height': 480,
    'img_width': 640,
    'input_channels': 1, # Grayscale
    
    # Stereo Geometry
    'max_disp_pixel': 192,         # Max disparity in original image pixels
    'backbone_stride': 4,          # MobileNet Stage 1 output
    'internal_disp_steps': 48,     # 192 / 4 = 48 steps for cost volume
    
    # Heads
    'num_seg_classes': 6,          # 5 Robotics + Background
    'num_det_classes': 80,         # COCO standard
    
    # Training
    'batch_size': 8,               # Adjust for 3080 Ti (12GB VRAM)
    'lr_backbone': 1e-4,
    'lr_heads': 3e-4,
}

print(f"ℹ️  Configuration Loaded. Max Disparity: {CONFIG['max_disp_pixel']}px ({CONFIG['internal_disp_steps']} internal steps)")

/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Running on: cuda (AMP Enabled)
ℹ️  Configuration Loaded. Max Disparity: 192px (48 internal steps)


In [2]:
# === CELL 2: DATA LOADING PIPELINE ===
class FusedHexapodDataset(Dataset):
    def __init__(self, root_dir, mode='train', task='stereo', transform=None):
        """
        mode: 'train' or 'val'
        task: 'stereo' (FT3D), 'robotics' (TartanAir), 'coco' (Detection)
        """
        self.root = root_dir
        self.mode = mode
        self.task = task
        self.transform = transform
        self.file_list = self._scan_files()

    def _scan_files(self):
        # Placeholder: Implement actual file scanning logic here
        # Return list of dicts: [{'left': path, 'right': path, 'disp': path, 'seg': path, ...}]
        return [] 

    def load_disp(self, path):
        # Handle .pfm or .png disparity loading
        if path.endswith('.pfm'):
            # Custom PFM loader would go here
            return np.zeros((480, 640), dtype=np.float32)
        return cv2.imread(path, cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0

    def __len__(self):
        return 100 # Dummy length for testing

    def __getitem__(self, idx):
        # 1. Load Grayscale Images
        # In real code: Load from self.file_list[idx]
        # Simulating data for architecture verification:
        left = torch.randn(1, 480, 640) 
        right = torch.randn(1, 480, 640)
        
        targets = {}
        
        if self.task == 'stereo':
            # GT Disparity
            targets['disp'] = torch.rand(480, 640) * 192.0
            targets['seg'] = None
            targets['det'] = None
            
        elif self.task == 'robotics':
            # TartanAir Seg + Det
            targets['disp'] = None # or sparse
            targets['seg'] = torch.randint(0, 6, (480, 640)).long()
            targets['det'] = torch.zeros((10, 5)) # [x,y,w,h,cls]
            
        return {'left': left, 'right': right, 'targets': targets}

# Test the shape
ds = FusedHexapodDataset("dummy", task='stereo')
sample = ds[0]
print(f"📦 Data Shape Check -> Left: {sample['left'].shape}")

📦 Data Shape Check -> Left: torch.Size([1, 480, 640])


In [3]:
# === CELL 3: MODEL ARCHITECTURE (FINAL STEREO) ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# --- Helper Blocks ---
class ConvMean(nn.Module):
    """ Replaces mean() with 1x1 Conv for NPU compatibility """
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, 1, bias=False)
        with torch.no_grad():
            self.conv.weight.fill_(1.0 / in_channels)
        self.conv.weight.requires_grad = False
    def forward(self, x): return self.conv(x)

class ResBlock(nn.Module):
    """ Standard ResBlock for Refinement """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

# --- Stereo Components ---
class SimpleCorrelation(nn.Module):
    """ Loop-based Correlation + Immediate Reduction (Low Memory) """
    def __init__(self, in_channels, max_disp):
        super().__init__()
        self.D = max_disp
        self.inter_ch = 16
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_ch, 1, bias=False),
            nn.BatchNorm2d(self.inter_ch), nn.ReLU(inplace=True)
        )
        self.reducer = ConvMean(self.inter_ch) 

    def forward(self, left, right):
        l, r = self.reduce(left), self.reduce(right)
        cost_stack = []
        for d in range(self.D):
            if d > 0:
                # Correlation: Left vs Right(shifted)
                # Pad LEFT to maintain W dimension (Standard stereo convention)
                sim = self.reducer(l[:,:,:,d:] * r[:,:,:,:-d])
                cost_stack.append(F.pad(sim, (d, 0, 0, 0)))
            else:
                cost_stack.append(self.reducer(l * r))
        return torch.cat(cost_stack, dim=1)

class GaussianGating(nn.Module):
    """ Gate = exp(-0.5 * (dist/sigma)^2) """
    def __init__(self, max_disp, gate_range=12):
        super().__init__()
        self.sigma = gate_range / 2.0
        self.disp_coords = nn.Parameter(
            torch.arange(max_disp).float().view(1, max_disp, 1, 1), 
            requires_grad=False
        )

    def forward(self, cost_volume, coarse_disp_s4):
        dist = self.disp_coords - coarse_disp_s4
        return cost_volume * torch.exp(-0.5 * (dist / self.sigma)**2)

# --- Main Stereo Head ---
class StereoHeadGatedV2(nn.Module):
    def __init__(self, ch_s16, ch_s4, max_disp_s4=48):
        super().__init__()
        self.D = max_disp_s4
        
        # 1. Coarse Stage (Stride 16)
        self.D_coarse = self.D // 4 # 12
        self.corr_coarse = SimpleCorrelation(ch_s16, self.D_coarse)
        self.refine_coarse = nn.Sequential(
            nn.Conv2d(self.D_coarse, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32), ResBlock(32),
            nn.Conv2d(32, self.D_coarse, 3, 1, 1, bias=False)
        )
        self.coarse_sum = nn.Conv2d(self.D_coarse, 1, 1, bias=False) 
        with torch.no_grad():
            self.coarse_sum.weight.data = torch.arange(self.D_coarse).float().view(1, -1, 1, 1)
        
        self.scale_bias = nn.Parameter(torch.zeros(1, 1, 1, 1)) # Learnable Correction

        # 2. Fine Stage (Stride 4)
        self.corr_fine = SimpleCorrelation(ch_s4, self.D)
        self.gating = GaussianGating(max_disp=self.D, gate_range=12) # Fixed +/- 12px
        self.refine_fine = nn.Sequential(
            nn.Conv2d(self.D, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32), ResBlock(32),
            nn.Conv2d(32, self.D, 3, 1, 1, bias=False) 
        )

    def forward(self, l16, r16, l4, r4):
        # Coarse
        c_cost = self.corr_coarse(l16, r16)
        c_cost = self.refine_coarse(c_cost)
        
        # Soft-Argmin (Temp=2.0)
        prob_c = F.softmax(c_cost * 2.0, dim=1)
        d_coarse_low = self.coarse_sum(prob_c)
        
        # Upsample & Bias Correct
        d_coarse_s4 = F.interpolate(d_coarse_low, size=l4.shape[-2:], mode='bilinear', align_corners=False) * 4.0
        d_coarse_s4 = d_coarse_s4 + self.scale_bias 
        
        # Fine
        raw_cost = self.corr_fine(l4, r4)
        gated_cost = self.gating(raw_cost, d_coarse_s4)
        final_cost = self.refine_fine(gated_cost)
        
        return c_cost, final_cost, d_coarse_s4

# --- The Fused Brain ---
class FusedHexapodModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Shared Backbone
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, features_only=True, out_indices=(1, 2, 3, 4))
        # Channels: S4=24, S8=40, S16=112, S32=960
        
        # Stereo Head
        self.stereo = StereoHeadGatedV2(ch_s16=112, ch_s4=24)
        
        # [FUTURE] Segmentation Head (LR-ASPP) will go here
        self.seg_head = nn.Identity() 

        # [FUTURE] YOLO Head will go here
        self.yolo_head = nn.Identity()

    def forward(self, left, right=None):
        x = left.repeat(1, 3, 1, 1) # Grayscale -> RGB
        
        # Extract Features [s4, s8, s16, s32]
        fl = self.backbone(x)
        
        # Stereo Branch
        c_log, f_log, d_c = None, None, None
        if right is not None:
            xr = right.repeat(1, 3, 1, 1)
            fr = self.backbone(xr)
            c_log, f_log, d_c = self.stereo(fl[2], fr[2], fl[0], fr[0])
            
        # [FUTURE] Seg Branch: seg_pred = self.seg_head(fl[0], fl[2])
        seg_pred = None
        
        # [FUTURE] Det Branch: det_preds = self.yolo_head(fl[1], fl[2], fl[3])
        det_pred = None
        
        return c_log, f_log, d_c, seg_pred, det_pred

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = FusedHexapodModel().to(DEVICE)
print("✅ Fused Model initialized. Stereo Module verified.")

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✅ Fused Model initialized. Stereo Module verified.


In [4]:
# === CELL 4: LOSS FUNCTIONS & POST-PROCESSING ===
import torch.optim as optim
from torch.cuda.amp import GradScaler

class StereoPostProcessor:
    """ Converts NPU Logits -> Disparity Map (CPU side) """
    @staticmethod
    def process(logits, max_disp_s4=48):
        prob = F.softmax(logits, dim=1) 
        coords = torch.arange(max_disp_s4, device=logits.device).float().view(1, -1, 1, 1)
        disp_s4 = torch.sum(prob * coords, dim=1, keepdim=True)
        # Scale 4.0 (Stride 4 -> Full Res)
        return F.interpolate(disp_s4, scale_factor=4, mode='bilinear', align_corners=False) * 4.0

class HexapodLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.processor = StereoPostProcessor()
        # [FUTURE] self.seg_loss = nn.CrossEntropyLoss(...)
        # [FUTURE] self.det_loss = ...
        
    def sobel(self, img):
        kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], device=img.device).float().view(1,1,3,3)
        ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], device=img.device).float().view(1,1,3,3)
        return torch.sqrt(F.conv2d(img, kx, padding=1)**2 + F.conv2d(img, ky, padding=1)**2 + 1e-6)

    def forward(self, preds, targets):
        c_log, f_log, d_coarse_s4, seg_p, det_p = preds
        t_disp = targets.get('disp')
        total_loss = 0; logs = {}
        
        # --- Stereo Loss ---
        if t_disp is not None and f_log is not None:
            if t_disp.dim() == 3:
                t_disp = t_disp.unsqueeze(1)
            mask = (t_disp > 0) & (t_disp < 192)
            if mask.sum() > 0:
                # 1. Fine Loss (L1 + Sobel)
                disp_pred = self.processor.process(f_log)
                loss_main = F.smooth_l1_loss(disp_pred[mask], t_disp[mask]) + \
                            0.5 * F.l1_loss(self.sobel(disp_pred)[mask], self.sobel(t_disp)[mask])
                
                # 2. Coarse Aux Loss (Supervise the Gating Signal)
                # d_coarse_s4 is already scaled to Stride 4 magnitude (0..48)
                # We interpolate to full res spatial size, then multiply by 4 to get full res magnitude (0..192)
                d_c_full = F.interpolate(d_coarse_s4, size=t_disp.shape[-2:], mode='bilinear', align_corners=False) * 4.0
                l_aux = F.smooth_l1_loss(d_c_full[mask], t_disp[mask])
                
                loss = loss_main + 0.4 * l_aux
                total_loss += loss
                logs['stereo'] = loss.item()
                logs['coarse_aux'] = l_aux.item()

        return total_loss, logs

criterion = HexapodLoss().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scaler = GradScaler()
print("✅ Loss Functions ready.")

✅ Loss Functions ready.


/tmp/ipykernel_495/5736194.py:58: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [5]:
# === CELL 5: TRAINING LOOP ===
def train_epoch(model, optimizer, epoch, stage):
    model.train()
    # Freeze Logic
    if stage == 'coarse':
        for p in model.stereo.refine_fine.parameters(): p.requires_grad = False
        model.stereo.scale_bias.requires_grad = False # Freeze bias in coarse stage
    else:
        for p in model.stereo.refine_fine.parameters(): p.requires_grad = True
        model.stereo.scale_bias.requires_grad = True

    # Dummy Data Loop (Replace with DataLoader)
    print(f"--- Epoch {epoch} Stage: {stage} ---")
    for i in range(5):
        l = torch.randn(2, 1, 480, 640).to(DEVICE)
        r = torch.randn(2, 1, 480, 640).to(DEVICE)
        t = {'disp': torch.abs(torch.randn(2, 480, 640).to(DEVICE)) * 192}
        
        optimizer.zero_grad()
        # Handle modern vs legacy AMP syntax
        try:
            autocast_ctx = torch.amp.autocast('cuda')
        except:
            autocast_ctx = torch.cuda.amp.autocast()
        with autocast_ctx:
            preds = model(l, r)
            loss, _ = criterion(preds, t)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if i==0: print(f"Loss: {loss.item():.4f}")

# Example Run
print("🚀 1. Train Coarse..."); train_epoch(model, optimizer, 1, 'coarse')
print("🚀 2. Train Fine..."); train_epoch(model, optimizer, 1, 'fine')

🚀 1. Train Coarse...
--- Epoch 1 Stage: coarse ---
Loss: 307.2415
🚀 2. Train Fine...
--- Epoch 1 Stage: fine ---
Loss: 306.2213


In [6]:
# === CELL 5.1: STEREO TEST TRAINING (SANITY CHECK) ===
import torch.optim as optim
import time

def run_sanity_check():
    print("🧪 Starting Stereo-Only Sanity Check...")
    
    # 1. Setup Model & Optimizer
    model.train()
    # Freezing logic for Coarse Stage (just to test that logic too)
    for p in model.stereo.refine_fine.parameters(): p.requires_grad = False
    model.stereo.scale_bias.requires_grad = False
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    scaler = torch.cuda.amp.GradScaler() # For Mixed Precision
    
    # 2. Dummy Data Generator (Simulates a Batch)
    # Batch=2, 1 Channel, 480x640
    dummy_l = torch.randn(2, 1, 480, 640).to(DEVICE)
    dummy_r = torch.randn(2, 1, 480, 640).to(DEVICE)
    # Dummy Disparity (0 to 192)
    dummy_disp = torch.abs(torch.randn(2, 480, 640).to(DEVICE)) * 192.0
    
    targets = {'disp': dummy_disp}
    
    # 3. Training Step Simulation
    start = time.time()
    
    # Forward
    optimizer.zero_grad()
    # Handle modern vs legacy AMP syntax
    try:
        autocast_ctx = torch.amp.autocast('cuda')
    except:
        autocast_ctx = torch.cuda.amp.autocast()
    with autocast_ctx:
        # The model returns 5 values (c, f, d, seg, det). Seg/Det are None.
        preds = model(dummy_l, dummy_r)
        
        # Loss checks only 'disp'
        loss, logs = criterion(preds, targets)
    
    # Backward
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    end = time.time()
    
    # 4. Report
    print(f"✅ Forward/Backward Pass Successful!")
    print(f"⏱️  Time per batch: {(end-start)*1000:.2f} ms")
    print(f"📉 Loss Value: {loss.item():.4f}")
    print(f"📝 Logs: {logs}")
    
    # Check outputs
    c_log, f_log, d_c, _, _ = preds
    print(f"📦 Output Shapes:")
    print(f"   - Coarse Logits: {c_log.shape} (Should be [2, 12, 30, 40])")
    print(f"   - Fine Logits:   {f_log.shape} (Should be [2, 48, 120, 160])")
    print(f"   - Coarse Disp:   {d_c.shape}   (Should be [2, 1, 120, 160])")

# Run it!
run_sanity_check()

🧪 Starting Stereo-Only Sanity Check...
✅ Forward/Backward Pass Successful!
⏱️  Time per batch: 169.22 ms
📉 Loss Value: 304.4653
📝 Logs: {'stereo': 304.46533203125, 'coarse_aux': 47.177772521972656}
📦 Output Shapes:
   - Coarse Logits: torch.Size([2, 12, 30, 40]) (Should be [2, 12, 30, 40])
   - Fine Logits:   torch.Size([2, 48, 120, 160]) (Should be [2, 48, 120, 160])
   - Coarse Disp:   torch.Size([2, 1, 120, 160])   (Should be [2, 1, 120, 160])


/tmp/ipykernel_495/2218452523.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For Mixed Precision


In [7]:
# === CELL 6: EXPORT ===
def export():
    model.eval()
    dummy = torch.randn(1, 1, 480, 640).to(DEVICE)
    try:
        torch.onnx.export(model, (dummy, dummy), "hexapod_final.onnx", 
                          input_names=['left', 'right'], 
                          output_names=['coarse_logits', 'fine_logits', 'coarse_disp'], 
                          opset_version=11)
        print("✅ Export Successful! Ready for Hailo.")
    except Exception as e: print(f"❌ Export Failed: {e}")
export()

/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:663: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:1186: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../to

✅ Export Successful! Ready for Hailo.


In [8]:
# === CELL 7: SEGMENTATION HEAD (LR-ASPP) ===
class LRASPPHead(nn.Module):
    """
    Lite R-ASPP Head (MobileNetV3 Standard).
    Fuses High-Level semantic features (Stride 16) with Low-Level details (Stride 4).
    """
    def __init__(self, low_channels, high_channels, num_classes):
        super().__init__()
        self.inter_channels = 128
        
        # 1. High-Level Branch (Stride 16)
        # 1x1 Conv -> BN -> ReLU
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_channels, self.inter_channels, 1, bias=False),
            nn.BatchNorm2d(self.inter_channels),
            nn.ReLU(inplace=True)
        )
        
        # Global Context Gating (Global Avg Pool -> Conv -> Sigmoid)
        self.scale_high = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(high_channels, self.inter_channels, 1, bias=False),
            nn.Sigmoid()
        )
        
        # 2. Low-Level Branch (Stride 4)
        # Project to num_classes directly (saving compute)
        self.low_classifier = nn.Conv2d(low_channels, num_classes, 1)
        
        # 3. Final Classifier
        self.high_classifier = nn.Conv2d(self.inter_channels, num_classes, 1)

    def forward(self, x_low, x_high):
        # x_low: [B, 24, H/4, W/4]
        # x_high: [B, 112, H/16, W/16]
        
        # A. Process High Level
        out = self.cbr_high(x_high)
        w = self.scale_high(x_high)
        out = out * w # Gating
        
        # Upsample High -> Low Resolution (x4)
        out = F.interpolate(out, size=x_low.shape[-2:], mode='bilinear', align_corners=False)
        
        # B. Process Low Level & Fuse
        return self.low_classifier(x_low) + self.high_classifier(out)

# --- UPDATE MODEL TO INCLUDE SEG HEAD ---
# We monkey-patch the model class to add the head (or you can edit Cell 3 directly)
model.seg_head = LRASPPHead(low_channels=24, high_channels=112, num_classes=6).to(DEVICE)

# Update Forward Pass Wrapper
def model_forward_with_seg(left, right=None):
    x = left.repeat(1, 3, 1, 1)
    fl = model.backbone(x)
    
    # Stereo
    c_log, f_log, d_c = None, None, None
    if right is not None:
        xr = right.repeat(1, 3, 1, 1)
        fr = model.backbone(xr)
        c_log, f_log, d_c = model.stereo(fl[2], fr[2], fl[0], fr[0])
        
    # Seg (Inputs: Stride 4 and Stride 16)
    seg_pred = model.seg_head(fl[0], fl[2])
    
    return c_log, f_log, d_c, seg_pred, None # Det is still None

# Bind new forward method
model.forward = model_forward_with_seg
print("✅ Segmentation Head Attached & Forward Logic Updated.")

✅ Segmentation Head Attached & Forward Logic Updated.


In [9]:
# === CELL 8: LOSS UPDATE (STEREO + SEG + DISTILLATION) ===
class HexapodLossV2(HexapodLoss): # Inherit to keep stereo logic
    def __init__(self):
        super().__init__()
        # Standard GT Loss (Ignore 'void' class 255)
        self.seg_ce = nn.CrossEntropyLoss(ignore_index=255)
        
        # Distillation Loss (KL Divergence)
        self.kl_div = nn.KLDivLoss(reduction='batchmean')
        self.temp = 4.0 # Temperature for Soft Labels

    def forward(self, preds, targets, teacher_preds=None):
        # Unpack
        c_log, f_log, d_coarse_s4, seg_p, det_p = preds
        t_disp = targets.get('disp')
        t_seg = targets.get('seg')
        
        total_loss = 0
        logs = {}
        
        # 1. RUN STEREO LOSS (Inherited Logic)
        # We manually call the parent logic parts or copy-paste for clarity.
        # Let's copy-paste the stereo part for safety to ensure it runs:
        if t_disp is not None and f_log is not None:
            if t_disp.dim() == 3: t_disp = t_disp.unsqueeze(1)
            mask = (t_disp > 0) & (t_disp < 192)
            if mask.sum() > 0:
                disp_pred = self.processor.process(f_log)
                l_main = F.smooth_l1_loss(disp_pred[mask], t_disp[mask]) + \
                         0.5 * F.l1_loss(self.sobel(disp_pred)[mask], self.sobel(t_disp)[mask])
                d_c_full = F.interpolate(d_coarse_s4, size=t_disp.shape[-2:], mode='bilinear') * 4.0
                l_aux = F.smooth_l1_loss(d_c_full[mask], t_disp[mask])
                
                l_stereo = l_main + 0.4 * l_aux
                total_loss += l_stereo
                logs['stereo'] = l_stereo.item()

        # 2. RUN SEGMENTATION LOSS
        if seg_p is not None:
            # Upsample Seg Logits to Full Res for Loss Calc
            # (Target is 480x640, Logits are 120x160)
            seg_full = F.interpolate(seg_p, size=(480, 640), mode='bilinear', align_corners=False)
            
            # A. Supervised GT Loss
            if t_seg is not None:
                l_seg_gt = self.seg_ce(seg_full, t_seg.long())
                total_loss += l_seg_gt
                logs['seg_gt'] = l_seg_gt.item()
            
            # B. Knowledge Distillation (If Teacher is present)
            if teacher_preds is not None and 'seg' in teacher_preds:
                t_logits = teacher_preds['seg']
                # Softmax with Temp
                student_soft = F.log_softmax(seg_full / self.temp, dim=1)
                teacher_soft = F.softmax(t_logits / self.temp, dim=1)
                
                l_kd = self.kl_div(student_soft, teacher_soft) * (self.temp ** 2)
                
                # KD usually weighted 0.5 vs GT
                total_loss += 0.5 * l_kd
                logs['seg_kd'] = l_kd.item()

        return total_loss, logs

# Hot-swap the criterion
criterion = HexapodLossV2().to(DEVICE)
print("✅ Loss V2 (Stereo + Seg + KD) Loaded.")

✅ Loss V2 (Stereo + Seg + KD) Loaded.


In [10]:
# === CELL 9: SEGMENTATION SANITY CHECK ===
def run_seg_check():
    print("🧪 Starting Seg+Stereo Check...")
    optimizer.zero_grad()
    
    # Dummy Data
    l = torch.randn(2, 1, 480, 640).to(DEVICE)
    r = torch.randn(2, 1, 480, 640).to(DEVICE)
    t_disp = torch.abs(torch.randn(2, 480, 640).to(DEVICE)) * 192.0
    # Dummy Seg Targets (Classes 0-5)
    t_seg = torch.randint(0, 6, (2, 480, 640)).to(DEVICE)
    
    targets = {'disp': t_disp, 'seg': t_seg}
    
    # Forward
    try:
        autocast_ctx = torch.amp.autocast('cuda')
    except:
        autocast_ctx = torch.cuda.amp.autocast()

    with autocast_ctx:
        preds = model(l, r)
        loss, logs = criterion(preds, targets)
        
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    # Check Seg Output Shape
    _, _, _, seg_p, _ = preds
    print(f"✅ Pass Successful! Loss: {loss.item():.4f}")
    print(f"📝 Logs: {logs}")
    print(f"📦 Seg Output Shape: {seg_p.shape} (Should be [2, 6, 120, 160])")

run_seg_check()

🧪 Starting Seg+Stereo Check...
✅ Pass Successful! Loss: 318.9215
📝 Logs: {'stereo': 314.3254699707031, 'seg_gt': 4.596011638641357}
📦 Seg Output Shape: torch.Size([2, 6, 120, 160]) (Should be [2, 6, 120, 160])


In [11]:
# === CELL 10: YOLO DETECTION HEAD (ANCHOR-FREE) ===
class DecoupledHead(nn.Module):
    """
    Standard YOLOv8-style Decoupled Head for one scale.
    Splits features into Box Regression branch and Class branch.
    """
    def __init__(self, in_channels, num_classes=80):
        super().__init__()
        inter_ch = max(in_channels, 64)
        
        # Stem (Reduce/Adjust channels)
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, inter_ch, 1, bias=False),
            nn.BatchNorm2d(inter_ch),
            nn.SiLU(inplace=True) # SiLU is standard for YOLO and NPU-friendly
        )
        
        # Classification Branch
        self.cls_branch = nn.Sequential(
            nn.Conv2d(inter_ch, inter_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(inter_ch), nn.SiLU(inplace=True),
            nn.Conv2d(inter_ch, inter_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(inter_ch), nn.SiLU(inplace=True),
            nn.Conv2d(inter_ch, num_classes, 1) # Output: Class Logits
        )
        
        # Box Regression Branch (Distribution Focal Loss style)
        # Outputs 4 values per pixel: dist to Left, Top, Right, Bottom
        self.reg_branch = nn.Sequential(
            nn.Conv2d(inter_ch, inter_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(inter_ch), nn.SiLU(inplace=True),
            nn.Conv2d(inter_ch, inter_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(inter_ch), nn.SiLU(inplace=True),
            nn.Conv2d(inter_ch, 4, 1) # Output: L, T, R, B offsets
        )

    def forward(self, x):
        x = self.stem(x)
        return self.cls_branch(x), self.reg_branch(x)

class YOLOHead(nn.Module):
    def __init__(self, ch_dims, num_classes=80):
        super().__init__()
        # ch_dims: [s4, s8, s16, s32]. We use indices 1, 2, 3.
        self.head_s8  = DecoupledHead(ch_dims[1], num_classes)
        self.head_s16 = DecoupledHead(ch_dims[2], num_classes)
        self.head_s32 = DecoupledHead(ch_dims[3], num_classes)

    def forward(self, x_s8, x_s16, x_s32):
        # Process each scale independently
        cls8, reg8  = self.head_s8(x_s8)
        cls16, reg16 = self.head_s16(x_s16)
        cls32, reg32 = self.head_s32(x_s32)
        
        # Return as list of tuples
        return [(cls8, reg8), (cls16, reg16), (cls32, reg32)]

# --- UPDATE MODEL ---
# 1. Initialize Head
# MobileNetV3 Large channels: S8=40, S16=112, S32=960 (check your specific timm model)
# We assume the standard config here.
model.yolo_head = YOLOHead(ch_dims=[24, 40, 112, 960], num_classes=80).to(DEVICE)

# 2. Update Forward Logic
def model_forward_final(left, right=None):
    x = left.repeat(1, 3, 1, 1)
    fl = model.backbone(x) # [s4, s8, s16, s32]
    
    # Stereo
    c_log, f_log, d_c = None, None, None
    if right is not None:
        xr = right.repeat(1, 3, 1, 1)
        fr = model.backbone(xr)
        c_log, f_log, d_c = model.stereo(fl[2], fr[2], fl[0], fr[0])
        
    # Seg
    seg_pred = model.seg_head(fl[0], fl[2])
    
    # Det (Inputs: S8, S16, S32)
    # fl[1]=40ch, fl[2]=112ch, fl[3]=960ch
    det_pred = model.yolo_head(fl[1], fl[2], fl[3])
    
    return c_log, f_log, d_c, seg_pred, det_pred

model.forward = model_forward_final
print("✅ YOLO Head Attached. Full Fused Stack Ready.")

✅ YOLO Head Attached. Full Fused Stack Ready.


In [26]:
# === CELL 13: FINAL ROBUST LOSS (AUTO-UPSAMPLING) ===
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleYOLOLoss(nn.Module):
    def __init__(self, num_classes=80):
        super().__init__()
        self.num_classes = num_classes
        
    def _get_targets(self, targets_list, cls_preds, reg_preds):
        batch_size = cls_preds.shape[0]
        device = cls_preds.device
        H, W = cls_preds.shape[2], cls_preds.shape[3]
        
        cls_targets = torch.zeros_like(cls_preds)
        reg_targets = torch.zeros_like(reg_preds)
        obj_mask = torch.zeros((batch_size, H, W), device=device)
        
        for b in range(batch_size):
            t = targets_list[b]
            if t is None or t.numel() == 0: continue
            
            # Clamp targets 0..1
            t[:, 1:5] = torch.clamp(t[:, 1:5], 0.0, 1.0)
            
            gt_cls = t[:, 0].long().clamp(0, self.num_classes - 1)
            gt_x = (t[:, 1] * W).long().clamp(0, W-1)
            gt_y = (t[:, 2] * H).long().clamp(0, H-1)
            gt_w = t[:, 3] * W
            gt_h = t[:, 4] * H
            
            for i in range(len(t)):
                x, y = gt_x[i], gt_y[i]
                cls_targets[b, gt_cls[i], y, x] = 1.0
                obj_mask[b, y, x] = 1.0
                reg_targets[b, 0, y, x] = gt_w[i] / 2.0
                reg_targets[b, 1, y, x] = gt_h[i] / 2.0
                reg_targets[b, 2, y, x] = gt_w[i] / 2.0
                reg_targets[b, 3, y, x] = gt_h[i] / 2.0
                
        return cls_targets, reg_targets, obj_mask

    def forward(self, preds, targets):
        cls_p, reg_p = preds[2] 
        if targets is None: return torch.tensor(0.0, device=cls_p.device)
        cls_t, reg_t, mask = self._get_targets(targets, cls_p, reg_p)
        
        loss_cls = F.binary_cross_entropy_with_logits(cls_p, cls_t)
        if mask.sum() > 0:
            mask_reg = mask.unsqueeze(1).repeat(1, 4, 1, 1).bool()
            loss_reg = F.smooth_l1_loss(reg_p[mask_reg], reg_t[mask_reg], beta=1.0)
        else:
            loss_reg = torch.tensor(0.0, device=cls_p.device)
        return loss_cls + loss_reg

class HexapodLossV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.yolo_loss = SimpleYOLOLoss()
        self.w_stereo = 1.0
        self.w_seg = 1.0
        self.w_kd = 1.0 
        self.w_yolo = 1.0
        self.T = 4.0

    def forward(self, preds, targets, teacher_preds=None):
        stereo_packet, s4, s8, seg_preds, det_preds = preds
        logs = {}
        total_loss = 0
        
        # --- 1. STEREO LOSS (WITH SOFT-ARGMAX) ---
        if targets.get('disp') is not None:
            gt_disp = targets['disp'] # [B, 1, H, W]
            
            # Extract raw stereo output
            raw_stereo = stereo_packet[-1] if isinstance(stereo_packet, list) else stereo_packet
            
            # --- CONVERT 12 CHANNELS TO 1 DISPARITY ---
            if raw_stereo.shape[1] > 1:
                # Soft-Argmax: Weighted average of indices
                # [B, 12, H, W] -> Softmax over dim 1
                probs = F.softmax(raw_stereo, dim=1)
                # Indices [0, 1, ..., 11]
                indices = torch.arange(raw_stereo.shape[1]).to(raw_stereo.device).view(1, -1, 1, 1).float()
                # Expected value: sum(prob * index)
                pred_disp = torch.sum(probs * indices, dim=1, keepdim=True)
            else:
                pred_disp = raw_stereo

            # --- UPSAMPLE & MASK ---
            if pred_disp.shape[-2:] != gt_disp.shape[-2:]:
                pred_disp = F.interpolate(pred_disp, size=gt_disp.shape[-2:], mode='bilinear')
            
            mask = (gt_disp > 0)
            if mask.sum() > 0:
                l_stereo = F.smooth_l1_loss(pred_disp[mask], gt_disp[mask], beta=1.0)
                total_loss += self.w_stereo * l_stereo
                logs['stereo'] = l_stereo.item()

        # --- 2. SEGMENTATION LOSS (AUTO-UPSAMPLE) ---
        if targets.get('seg') is not None and seg_preds is not None:
            gt_seg = targets['seg']
            seg_preds_up = F.interpolate(seg_preds, size=gt_seg.shape[-2:], mode='bilinear')
            l_seg = F.cross_entropy(seg_preds_up, gt_seg, ignore_index=255)
            total_loss += self.w_seg * l_seg
            logs['seg_gt'] = l_seg.item()

        # --- 3. DISTILLATION LOSS ---
        if teacher_preds is not None and seg_preds is not None:
            t_logits = teacher_preds['seg']
            s_logits = F.interpolate(seg_preds, size=t_logits.shape[-2:], mode='bilinear')
            p = F.softmax(t_logits / self.T, dim=1)
            q = F.log_softmax(s_logits / self.T, dim=1)
            kl = F.kl_div(q, p, reduction='sum')
            l_kd = (kl / (s_logits.shape[0] * s_logits.shape[2] * s_logits.shape[3])) * (self.T ** 2)
            total_loss += self.w_kd * l_kd
            logs['seg_kd'] = l_kd.item()

        # --- 4. YOLO LOSS ---
        if targets.get('det') is not None and det_preds is not None:
            l_yolo = self.yolo_loss(det_preds, targets['det'])
            total_loss += self.w_yolo * l_yolo
            logs['yolo'] = l_yolo.item()
            
        return total_loss, logs

criterion = HexapodLossV3().to(DEVICE)
print("✅ Stereo-Fixed Loss Loaded. 'stereo' loss should now appear.")

✅ Stereo-Fixed Loss Loaded. 'stereo' loss should now appear.


In [13]:
# === CELL 12: FULL STACK SANITY CHECK ===
def run_full_check():
    print("🧪 Starting Full Stack (Stereo + Seg + YOLO) Check...")
    optimizer.zero_grad()
    
    # Inputs
    l = torch.randn(2, 1, 480, 640).to(DEVICE)
    r = torch.randn(2, 1, 480, 640).to(DEVICE)
    
    # Targets
    t_disp = torch.abs(torch.randn(2, 480, 640).to(DEVICE)) * 192.0
    t_seg = torch.randint(0, 6, (2, 480, 640)).to(DEVICE)
    t_det_dummy = torch.randn(2, 10, 5).to(DEVICE) # Dummy boxes
    
    targets = {'disp': t_disp, 'seg': t_seg, 'det_sanity': t_det_dummy}
    
    # Forward
    try:
        autocast_ctx = torch.amp.autocast('cuda')
    except:
        autocast_ctx = torch.cuda.amp.autocast()

    with autocast_ctx:
        preds = model(l, r)
        loss, logs = criterion(preds, targets)
        
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    # Verify Outputs
    _, _, _, _, det_p = preds
    print(f"✅ Pass Successful! Total Loss: {loss.item():.4f}")
    print(f"📝 Logs: {logs}")
    print(f"📦 YOLO Output (Scale 0) Shapes:")
    print(f"   - Cls: {det_p[0][0].shape} (Should be [2, 80, 60, 80])")
    print(f"   - Reg: {det_p[0][1].shape} (Should be [2, 4, 60, 80])")

run_full_check()

🧪 Starting Full Stack (Stereo + Seg + YOLO) Check...
✅ Pass Successful! Total Loss: 4.6022
📝 Logs: {'seg_gt': 4.602196216583252}
📦 YOLO Output (Scale 0) Shapes:
   - Cls: torch.Size([2, 80, 60, 80]) (Should be [2, 80, 60, 80])
   - Reg: torch.Size([2, 4, 60, 80]) (Should be [2, 4, 60, 80])


In [14]:
# === CELL 14: LOAD SEGMENTATION TEACHER ===
from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights

class SegmentationTeacher(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        # Load heavy ResNet101-based DeepLab
        # We start with COCO weights for the backbone, but replace the classifier
        weights = DeepLabV3_ResNet101_Weights.DEFAULT
        self.model = deeplabv3_resnet101(weights=weights)
        
        # Replace Head to match our 6 Robotics Classes
        # (In real training, you'd load a checkpoint where this Teacher 
        #  was already trained on your specific Seg dataset)
        self.model.classifier[4] = nn.Conv2d(256, num_classes, 1)
        self.model.aux_classifier[4] = nn.Conv2d(256, num_classes, 1)
        
        # Freeze parameters (Teacher should not update)
        for p in self.model.parameters():
            p.requires_grad = False
        self.model.eval()

    def forward(self, x):
        # DeepLab expects normalized inputs, handled by Dataset usually.
        # Returns OrderedDict( ['out': logits, 'aux': logits] )
        with torch.no_grad():
            return self.model(x)['out']

# Initialize Teacher
# Move to GPU, but keep in Eval mode
teacher_seg = SegmentationTeacher(num_classes=6).to(DEVICE)

print("✅ Segmentation Teacher (DeepLabV3+ ResNet101) Loaded & Frozen.")

✅ Segmentation Teacher (DeepLabV3+ ResNet101) Loaded & Frozen.


In [15]:
# === CELL 15: DISTILLATION TRAINING STEP (CORRECTED) ===
def train_distilled_batch(images_l, images_r, images_teacher, targets):
    """
    images_l: [B, 1, H, W] - Grayscale for Student (Robot)
    images_r: [B, 1, H, W] - Grayscale for Student (Robot)
    images_teacher: [B, 3, H, W] - RGB/Repeated Grayscale for Teacher
    """
    
    # 1. Teacher Pass (No Grad)
    with torch.no_grad():
        # Teacher uses the specific 3-channel input
        t_seg_logits = teacher_seg(images_teacher)
        
        # Pack into dictionary for Loss function
        teacher_preds = {'seg': t_seg_logits}
    
    # 2. Student Pass (Grad)
    optimizer.zero_grad()
    
    try:
        autocast_ctx = torch.amp.autocast('cuda')
    except:
        autocast_ctx = torch.cuda.amp.autocast()

    with autocast_ctx:
        # Student uses the 1-channel input (and internally repeats it)
        preds = model(images_l, images_r)
        
        # 3. Calculate Loss with Distillation
        loss, logs = criterion(preds, targets, teacher_preds=teacher_preds)
        
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    return loss.item(), logs

print("✅ Distillation Training Step defined (Inputs aligned).")

✅ Distillation Training Step defined (Inputs aligned).


In [16]:
# === CELL 16: TEACHER CONNECTIVITY CHECK (FIXED) ===
def run_teacher_check():
    print("🧪 Starting Distillation Check...")
    
    # 1. Inputs
    l = torch.randn(2, 1, 480, 640).to(DEVICE)
    r = torch.randn(2, 1, 480, 640).to(DEVICE)
    teacher_img = l.repeat(1, 3, 1, 1) 
    
    # 2. Dummy Targets (Fixed Boxes)
    # Boxes: [Batch, N, 5] -> [2, 10, 5]
    # Format: [class, x_center, y_center, w, h] (Normalized 0..1)
    # FIX: Use torch.rand for 0..1 uniform distribution
    dummy_boxes = torch.rand(2, 10, 5).to(DEVICE)
    dummy_boxes[:, :, 0] = 0 # Force class 0
    
    targets = {
        'disp': torch.abs(torch.randn(2, 480, 640).to(DEVICE)) * 192.0,
        'seg': torch.randint(0, 6, (2, 480, 640)).to(DEVICE),
        'det': dummy_boxes 
    }
    
    # Run
    loss, logs = train_distilled_batch(l, r, teacher_img, targets)
    
    print(f"✅ Distillation Pass Successful! Loss: {loss:.4f}")
    print(f"📝 Logs: {logs}")
    
    if 'seg_kd' in logs:
        print(f"🎉 SUCCESS: 'seg_kd' loss is present: {logs['seg_kd']:.4f}")

run_teacher_check()

🧪 Starting Distillation Check...
✅ Distillation Pass Successful! Loss: 14.3953
📝 Logs: {'seg_gt': 4.589151382446289, 'seg_kd': 4.803399562835693, 'yolo': 5.002715110778809}
🎉 SUCCESS: 'seg_kd' loss is present: 4.8034


In [17]:
# === CELL 14: FUSED DATASET (MONOCHROME / MIXED SOURCES) ===
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FusedMonoDataset(Dataset):
    def __init__(self, roots, mode='train', img_size=(480, 640)):
        """
        roots: dict with paths {'ft3d': 'path/to/ft3d', 'coco': 'path/to/coco', 'tartan': 'path/to/tartan'}
        mode: 'train' or 'val'
        """
        self.img_h, self.img_w = img_size
        self.mode = mode
        self.samples = []
        
        # --- 1. Load File Lists (Simplified for demonstration) ---
        # In practice, you would scan folders here. 
        # For now, we assume you have lists of (left_path, right_path, disp_path, seg_path, box_path)
        
        # A. Stereo Data (FT3D / TartanAir)
        # We flag these as type 'stereo'
        if 'tartan' in roots and os.path.exists(roots['tartan']):
            # Scan TartanAir folders...
            # self.samples.append({
            #    'type': 'stereo', 'l': l_path, 'r': r_path, 'd': disp_path, 's': seg_path
            # })
            pass 

        # B. Detection Data (COCO)
        # We flag these as type 'mono'
        if 'coco' in roots and os.path.exists(roots['coco']):
            # Scan COCO folders...
            # self.samples.append({
            #    'type': 'mono', 'l': img_path, 'b': box_label_path
            # })
            pass
            
        # [Placeholder] For sanity check, we create dummy samples in __getitem__ if list empty
        if len(self.samples) == 0:
            print("⚠️ No data found on disk. Generating synthetic samples for testing.")
            for _ in range(100):
                self.samples.append({'type': 'stereo'}) # Dummy mix
                self.samples.append({'type': 'mono'})

        # --- 2. Augmentations (Grayscale Enforced) ---
        # We use Albumentations to handle geometric transforms for both Img and Mask/Box
        self.transform = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            
            # CRITICAL: Convert everything to Grayscale first
            A.ToGray(p=1.0),
            
            # Geometric Augs (Shift/Scale/Rotate)
            # Note: We must be careful with Stereo. 
            # Vertical shift kills stereo correspondence! Horizontal shift changes disparity!
            # For simplicity in this fused loader, we disable geometric augs for Stereo samples
            # inside __getitem__, or keep them very safe (color jitter only).
            
            # Normalize to ImageNet mean (Grayscale equivalent)
            # RGB Mean [0.485, 0.456, 0.406] -> Avg is approx 0.45
            A.Normalize(mean=(0.45,), std=(0.225,), max_pixel_value=255.0),
            
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # --- DUMMY DATA GENERATION (Replace with cv2.imread) ---
        # Create random noise to simulate loading an image
        # Real code: img = cv2.imread(sample['l'], cv2.IMREAD_GRAYSCALE)
        raw_l = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8) # Start RGB (Simulate COCO)
        
        if sample['type'] == 'stereo':
            # Stereo Sample
            raw_r = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
            raw_d = np.random.rand(480, 640).astype(np.float32) * 192.0 # Disparity
            raw_s = np.random.randint(0, 6, (480, 640), dtype=np.uint8) # Seg
            
            # Apply Transforms (Replicated to Right image manually if needed)
            # For simplicity, we just apply TO_GRAY and Normalize here
            aug_l = self.transform(image=raw_l, mask=raw_d)['image'] # [1, 480, 640]
            aug_r = self.transform(image=raw_r)['image']             # [1, 480, 640]
            
            # Targets
            t_disp = torch.tensor(raw_d).unsqueeze(0) # [1, H, W]
            t_seg = torch.tensor(raw_s).long()
            t_det = torch.zeros((0, 5)) # No box targets for FT3D usually
            
            # Teacher Input: Needs 3 Channels (Fake RGB)
            teacher_input = aug_l.repeat(3, 1, 1)

        else:
            # Mono (COCO) Sample
            aug_l = self.transform(image=raw_l)['image']
            
            # Fake Right Image (Copy Left) -> Disparity will be 0
            aug_r = aug_l.clone()
            
            # Targets
            t_disp = torch.zeros((1, 480, 640)) - 1.0 # Mask out (-1)
            t_seg = torch.zeros((480, 640)).long() + 255 # Mask out (255)
            
            # Fake Boxes [cls, x, y, w, h]
            # Real code: load from txt/json
            t_det = torch.tensor([[1.0, 0.5, 0.5, 0.2, 0.2]]) # One dummy box
            
            teacher_input = aug_l.repeat(3, 1, 1)

        return {
            'left': aug_l,           # [1, H, W] - For Robot
            'right': aug_r,          # [1, H, W] - For Robot
            'teacher': teacher_input,# [3, H, W] - For DeepLab
            'disp': t_disp,
            'seg': t_seg,
            'det': t_det
        }

# Test the Loader
ds = FusedMonoDataset(roots={}, mode='train')
loader = DataLoader(ds, batch_size=2, shuffle=True)
batch = next(iter(loader))

print(f"✅ Fused Monochrome Loader Test:")
print(f"   Robot Input (Left): {batch['left'].shape} (Should be [2, 1, 480, 640])")
print(f"   Teacher Input:      {batch['teacher'].shape} (Should be [2, 3, 480, 640])")
print(f"   Stereo Target:      {batch['disp'].shape}")

⚠️ No data found on disk. Generating synthetic samples for testing.
✅ Fused Monochrome Loader Test:
   Robot Input (Left): torch.Size([2, 3, 480, 640]) (Should be [2, 1, 480, 640])
   Teacher Input:      torch.Size([2, 9, 480, 640]) (Should be [2, 3, 480, 640])
   Stereo Target:      torch.Size([2, 1, 480, 640])


In [18]:
# === CELL 17: REAL DATA PATH VERIFICATION ===
import os
import glob

# UPDATE THESE PATHS TO MATCH YOUR UNPACKING LOCATION
DATA_ROOTS = {
    'coco':   '../datasets/coco',            # Expects 'train2017' inside
    'ft3d':   '../datasets/FlyingThings3D',  # Expects 'frames_cleanpass' inside
    'tartan': '../datasets/TartanAir'        # Expects 'office', 'neighborhood' etc.
}

def verify_data_structure():
    print("📂 Scanning Dataset Directories...")
    valid_counts = {'coco': 0, 'ft3d': 0, 'tartan': 0}
    
    # 1. COCO Check
    coco_train = os.path.join(DATA_ROOTS['coco'], 'train2017')
    if os.path.exists(coco_train):
        # fast count
        num = len(os.listdir(coco_train))
        print(f"   ✅ COCO Train found: {num} images")
        valid_counts['coco'] = num
    else:
        print(f"   ❌ COCO Train missing at: {coco_train}")

    # 2. FlyingThings3D Check
    ft3d_img = os.path.join(DATA_ROOTS['ft3d'], 'frames_cleanpass/TRAIN')
    if os.path.exists(ft3d_img):
        # Check depth 2 subfolders (A/B/C...)
        # Just check one to be sure
        sub = os.listdir(ft3d_img)[0]
        sample = os.path.join(ft3d_img, sub, 'left')
        if os.path.exists(sample):
             print(f"   ✅ FT3D Train structure looks correct.")
             valid_counts['ft3d'] = 1
        else:
             print(f"   ⚠️ FT3D folder exists but sub-structure seems wrong (expected .../TRAIN/A/left)")
    else:
        print(f"   ❌ FT3D Train missing at: {ft3d_img}")

    # 3. TartanAir Check
    tartan_root = DATA_ROOTS['tartan']
    if os.path.exists(tartan_root):
        # Look for ANY environment folder
        envs = [d for d in os.listdir(tartan_root) if os.path.isdir(os.path.join(tartan_root, d))]
        if len(envs) > 0:
            print(f"   ✅ TartanAir found environments: {envs}")
            # Check for 'image_left' in the first env
            # Usually: office/Easy/P000/image_left
            sample_env = os.path.join(tartan_root, envs[0])
            # Recursive glob to find a left image folder
            left_folders = glob.glob(os.path.join(sample_env, '**', 'image_left'), recursive=True)
            if len(left_folders) > 0:
                 print(f"      - Verified 'image_left' folders exist.")
                 valid_counts['tartan'] = 1
            else:
                 print(f"      ⚠️ Environment found but 'image_left' subfolders missing. Check Unzip level.")
        else:
             print(f"   ⚠️ TartanAir root exists but is empty.")
    else:
        print(f"   ❌ TartanAir missing at: {tartan_root}")

    return valid_counts

# Run Verification
counts = verify_data_structure()

📂 Scanning Dataset Directories...
   ✅ COCO Train found: 118287 images
   ✅ FT3D Train structure looks correct.
   ✅ TartanAir found environments: ['neighborhood', 'office']
      - Verified 'image_left' folders exist.


In [19]:
# === CELL 18: DATASET PARSING HELPERS (PERMISSIVE) ===
import os
import glob
import json
import numpy as np
from pycocotools.coco import COCO

def parse_ft3d(root):
    """ Scans FT3D """
    print("   Scanning FlyingThings3D...")
    samples = []
    img_dir = os.path.join(root, 'frames_cleanpass/TRAIN/image_clean/left')
    disp_dir = os.path.join(root, 'disparity/TRAIN/disparity/left')
    
    if not os.path.exists(img_dir): return []
        
    left_files = sorted(glob.glob(os.path.join(img_dir, '*.png')))
    for l_path in left_files:
        filename = os.path.basename(l_path)
        r_path = l_path.replace('/left/', '/right/')
        d_path = os.path.join(disp_dir, filename.replace('.png', '.pfm'))
        if not os.path.exists(d_path): d_path = os.path.join(disp_dir, filename)
            
        if os.path.exists(r_path) and os.path.exists(d_path):
            samples.append({'type':'stereo','source':'ft3d','l':l_path,'r':r_path,'d':d_path,'s':None,'b':None})
            
    print(f"   -> Found {len(samples)} FT3D pairs.")
    return samples

def parse_tartan(root):
    """ Scans TartanAir (Segmentation Optional) """
    print("   Scanning TartanAir...")
    samples = []
    left_folders = glob.glob(os.path.join(root, '**', 'image_left'), recursive=True)
    
    for l_folder in left_folders:
        parent = os.path.dirname(l_folder)
        files = os.listdir(l_folder)
        
        for f in files:
            if not f.endswith('.png'): continue
            
            l_path = os.path.join(l_folder, f)
            r_filename = f.replace('_left', '_right')
            r_path = os.path.join(parent, 'image_right', r_filename)
            
            d_filename = f.replace('.png', '_depth.npy')
            d_path = os.path.join(parent, 'depth_left', d_filename)
            
            # Segmentation is OPTIONAL now
            s_filename = f.replace('.png', '_seg.npy')
            s_path = os.path.join(parent, 'seg_left', s_filename)
            if not os.path.exists(s_path): s_path = None
            
            # Check Critical Files (Img + Depth)
            # If Depth is missing, try .png
            if not os.path.exists(d_path):
                d_path_png = d_path.replace('.npy', '.png')
                if os.path.exists(d_path_png): d_path = d_path_png
                else: continue # Skip if no depth

            if os.path.exists(r_path):
                samples.append({
                    'type': 'stereo',
                    'source': 'tartan',
                    'l': l_path, 'r': r_path, 'd': d_path, 's': s_path, 'b': None
                })
        
        if len(samples) > 50000: break # Safety cap

    print(f"   -> Found {len(samples)} TartanAir samples.")
    return samples

def parse_coco(root):
    """ Scans COCO (Unchanged) """
    print("   Scanning COCO 2017...")
    samples = []
    ann_file = os.path.join(root, 'annotations', 'instances_train2017.json')
    img_dir = os.path.join(root, 'train2017')
    if not os.path.exists(ann_file): return []
    
    coco = COCO(ann_file)
    for iid in coco.getImgIds():
        img_info = coco.loadImgs(iid)[0]
        path = os.path.join(img_dir, img_info['file_name'])
        if not os.path.exists(path): continue
        anns = coco.loadAnns(coco.getAnnIds(imgIds=iid))
        boxes = []
        for ann in anns:
            if ann['iscrowd']: continue
            x, y, w, h = ann['bbox']
            H, W = img_info['height'], img_info['width']
            boxes.append([ann['category_id'] - 1, (x+w/2)/W, (y+h/2)/H, w/W, h/H])
        if len(boxes) > 0:
            samples.append({'type':'mono','source':'coco','l':path,'b':np.array(boxes, dtype=np.float32),'r':None,'d':None,'s':None})
            
    print(f"   -> Found {len(samples)} COCO samples.")
    return samples

print("✅ Parsers Fixed (Permissive Mode).")

✅ Parsers Fixed (Permissive Mode).


In [ ]:
# === CELL 19: REAL FUSED DATASET (ROBUST BOX FIX) ===
import re
import cv2
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset

def read_pfm(file):
    file = open(file, 'rb')
    header = file.readline().decode().rstrip()
    if header == 'PF': color = True
    elif header == 'Pf': color = False
    else: raise Exception('Not a PFM file.')
    
    dim_match = re.match(r'^(\d+)\s(\d+)\s$', file.readline().decode('utf-8'))
    if dim_match: width, height = map(int, dim_match.groups())
    else: raise Exception('Malformed PFM header.')
    
    scale = float(file.readline().decode().rstrip())
    if scale < 0: 
        endian = '<'
        scale = -scale
    else: 
        endian = '>'
    
    data = np.fromfile(file, endian + 'f')
    shape = (height, width, 3) if color else (height, width)
    
    data = np.reshape(data, shape) * scale
    return np.flipud(data)

class RealFusedDataset(Dataset):
    def __init__(self, roots, mode='train', img_size=(480, 640)):
        self.img_h, self.img_w = img_size
        self.samples = []
        
        # 1. Build Master List
        # (Assuming parse_ft3d, parse_tartan, parse_coco are defined in previous cells)
        if 'ft3d' in roots:   self.samples.extend(parse_ft3d(roots['ft3d']))
        if 'tartan' in roots: self.samples.extend(parse_tartan(roots['tartan']))
        if 'coco' in roots:   self.samples.extend(parse_coco(roots['coco']))
        
        print(f"📊 Total Training Samples: {len(self.samples)}")
        
        # 2. Augmentations
        # A. Main Transform (Left Image + Boxes + Masks)
        self.transform_main = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
        
        # B. Pure Transform (Right Image - No Boxes needed)
        self.transform_pure = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        
        # --- LOAD IMAGES (BGR) ---
        img_l_bgr = cv2.imread(s['l'])
        if img_l_bgr is None: 
            # Fallback for broken paths
            print(f"⚠️ Warning: Could not read {s['l']}. Returning random noise.")
            img_l_bgr = np.zeros((480, 640, 3), dtype=np.uint8)
            
        img_l_rgb = cv2.cvtColor(img_l_bgr, cv2.COLOR_BGR2RGB) # Teacher needs RGB
        
        # --- PREPARE TARGETS (ORIGINAL RESOLUTION) ---
        # 1. Disparity
        if s['d'] is not None:
            if s['d'].endswith('.pfm'):
                disp = read_pfm(s['d']).copy()
            elif s['d'].endswith('.npy'):
                disp = np.load(s['d'])
            disp = np.ascontiguousarray(disp)
        else:
            h, w = img_l_rgb.shape[:2]
            disp = np.zeros((h, w), dtype=np.float32) - 1.0 

        # 2. Segmentation
        if s['s'] is not None:
            seg = np.load(s['s'])
            seg = np.clip(seg, 0, 5)
        else:
            h, w = img_l_rgb.shape[:2]
            seg = np.zeros((h, w), dtype=np.uint8) + 255 # Ignore

        # 3. Boxes (WITH SANITIZATION)
        boxes = []
        class_labels = []
        if s['b'] is not None:
            for b in s['b']:
                # b is [class, x_center, y_center, w, h]
                cls_id = b[0]
                bx, by, bw, bh = b[1], b[2], b[3], b[4]
                
                # --- SANITY CHECK ---
                # Filter out boxes with 0 or negative area
                if bw <= 1e-4 or bh <= 1e-4: 
                    continue
                
                # Clamp to be safe (0.0 to 1.0)
                bx = np.clip(bx, 0, 1)
                by = np.clip(by, 0, 1)
                bw = np.clip(bw, 0, 1)
                bh = np.clip(bh, 0, 1)
                
                class_labels.append(cls_id)
                boxes.append([bx, by, bw, bh])
        
        # --- APPLY MAIN TRANSFORM (LEFT) ---
        masks = [disp, seg]
        
        try:
            transformed = self.transform_main(
                image=img_l_rgb, 
                masks=masks, 
                bboxes=boxes, 
                class_labels=class_labels
            )
        except ValueError as e:
            # Emergency fallback if Albumentations still fails
            # print(f"⚠️ Augmentation Error at idx {idx}: {e}. Returning empty boxes.")
            transformed = self.transform_main(
                image=img_l_rgb, 
                masks=masks, 
                bboxes=[], 
                class_labels=[]
            )
        
        teacher_input = transformed['image']
        t_disp = transformed['masks'][0].clone().detach().unsqueeze(0) 
        t_seg = transformed['masks'][1].clone().detach().long()       
        
        # Robot Input (Grayscale)
        robot_input = (teacher_input[0]*0.299 + teacher_input[1]*0.587 + teacher_input[2]*0.114).unsqueeze(0)
        
        # Boxes (Tensor)
        if len(transformed['bboxes']) > 0:
            b_list = []
            for c, b in zip(transformed['class_labels'], transformed['bboxes']):
                b_list.append([c] + list(b))
            t_det = torch.tensor(b_list, dtype=torch.float32)
        else:
            t_det = torch.zeros((0, 5), dtype=torch.float32)

        # --- APPLY PURE TRANSFORM (RIGHT) ---
        if s['type'] == 'stereo':
            img_r_bgr = cv2.imread(s['r'])
            if img_r_bgr is not None:
                img_r_rgb = cv2.cvtColor(img_r_bgr, cv2.COLOR_BGR2RGB)
                aug_r = self.transform_pure(image=img_r_rgb)['image']
                robot_right = (aug_r[0]*0.299 + aug_r[1]*0.587 + aug_r[2]*0.114).unsqueeze(0)
            else:
                robot_right = robot_input.clone()
        else:
            robot_right = robot_input.clone()

        return {
            'left': robot_input,
            'right': robot_right,
            'teacher': teacher_input,
            'disp': t_disp,
            'seg': t_seg,
            'det': t_det
        }

print("✅ RealFusedDataset Loaded (Robust Box Fix Applied).")

✅ RealFusedDataset Class Updated (Dual Transforms Fixed).


In [21]:
# === CELL 20: TEST DISK I/O (CORRECTED) ===

# EXACT PATHS FROM YOUR DIAGNOSTIC REPORT
REAL_ROOTS = {
    'coco':   '../datasets/coco',           # Lowercase 'coco'
    'ft3d':   '../datasets/FlyingThings3D', 
    'tartan': '../datasets/TartanAir'
}

# Init Dataset
print("🚀 Initializing Real Dataset...")
real_ds = RealFusedDataset(REAL_ROOTS, mode='train')

if len(real_ds) > 0:
    # Fetch one sample to ensure cv2.imread works
    sample = real_ds[0]
    print("\n✅ Sample Load Success!")
    print(f"   Source:     {real_ds.samples[0]['source']}")
    print(f"   Left Shape: {sample['left'].shape} (Grayscale)")
    print(f"   Teacher:    {sample['teacher'].shape} (RGB)")
    if sample['disp'] is not None:
        print(f"   Disp Mean:  {sample['disp'].mean():.2f}")
else:
    print("\n❌ Dataset is still empty! Check parsers.")

🚀 Initializing Real Dataset...
   Scanning FlyingThings3D...
   -> Found 21818 FT3D pairs.
   Scanning TartanAir...
   -> Found 38603 TartanAir samples.
   Scanning COCO 2017...
loading annotations into memory...
Done (t=10.72s)
creating index...
index created!
   -> Found 117266 COCO samples.
📊 Total Training Samples: 177687


NameError: name 're' is not defined

In [22]:
# === CELL 21: PATH DEBUGGER ===
import os

# Your defined roots
REAL_ROOTS = {
    'coco':   '../datasets/coco',
    'ft3d':   '../datasets/FlyingThings3D',
    'tartan': '../datasets/TartanAir'
}

def print_tree(startpath, depth=3):
    print(f"📂 Scanning: {startpath}")
    if not os.path.exists(startpath):
        print(f"   ❌ Path does not exist!")
        return

    prefix = 0
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        if level >= depth: continue
        
        indent = ' ' * 4 * (level)
        print(f"{indent}|-- {os.path.basename(root)}/")
        
        # Print specific key files we are looking for
        if 'instances_train2017.json' in files:
            print(f"{indent}    ⭐ FOUND: instances_train2017.json")
        if 'left' in dirs:
            print(f"{indent}    ⭐ FOUND: 'left' folder containing {len(os.listdir(os.path.join(root, 'left')))} images")
        if 'image_left' in dirs:
            print(f"{indent}    ⭐ FOUND: 'image_left' folder")
            
        sub_indent = ' ' * 4 * (level + 1)
        # Show first 2 files as examples
        for f in files[:2]:
            print(f"{sub_indent}|-- {f}")
        if len(files) > 2:
            print(f"{sub_indent}|-- ... ({len(files)-2} more files)")

print("--- DIAGNOSTIC REPORT ---")
print_tree(REAL_ROOTS['coco'], depth=2)
print("\n" + "="*40 + "\n")
print_tree(REAL_ROOTS['ft3d'], depth=4)
print("\n" + "="*40 + "\n")
print_tree(REAL_ROOTS['tartan'], depth=4)

--- DIAGNOSTIC REPORT ---
📂 Scanning: ../datasets/coco
|-- coco/
    |-- annotations/
        ⭐ FOUND: instances_train2017.json
        |-- instances_train2017.json
        |-- captions_val2017.json
        |-- ... (4 more files)
    |-- val2017/
        |-- 000000191761.jpg
        |-- 000000566436.jpg
        |-- ... (4998 more files)
    |-- train2017/
        |-- 000000130834.jpg
        |-- 000000064765.jpg
        |-- ... (118285 more files)


📂 Scanning: ../datasets/FlyingThings3D
|-- FlyingThings3D/
    |-- disparity/
        |-- TRAIN/
            |-- disparity/
                ⭐ FOUND: 'left' folder containing 21818 images
        |-- VAL/
            |-- disparity/
                ⭐ FOUND: 'left' folder containing 4248 images
    |-- frames_cleanpass/
        |-- TRAIN/
            |-- image_clean/
                ⭐ FOUND: 'left' folder containing 21818 images
        |-- .ipynb_checkpoints/
        |-- VAL/
            |-- image_clean/
                ⭐ FOUND: 'left' folder

In [23]:
# === CELL 22: TARTANAIR AUTO-UNZIPPER ===
import os
import zipfile
import glob
from tqdm import tqdm

# Path from your diagnostic report
TARTAN_ROOT = '../datasets/TartanAir'

def unzip_tartanair():
    print(f"🕵️ Scanning {TARTAN_ROOT} for zip files...")
    
    # Find all zip files recursively
    zips = glob.glob(os.path.join(TARTAN_ROOT, '**', '*.zip'), recursive=True)
    
    if len(zips) == 0:
        print("✅ No zip files found! (Maybe already unzipped?)")
        return

    print(f"📦 Found {len(zips)} zip files. Starting extraction...")
    
    for zip_path in tqdm(zips):
        target_dir = os.path.dirname(zip_path)
        
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                # Check if we already extracted (check first file)
                first_file = zip_ref.namelist()[0]
                if os.path.exists(os.path.join(target_dir, first_file)):
                    # Skip if already exists
                    continue
                
                # Extract
                zip_ref.extractall(target_dir)
                
        except zipfile.BadZipFile:
            print(f"❌ Corrupt Zip: {zip_path}")
        except Exception as e:
            print(f"⚠️ Error on {zip_path}: {e}")

    print("🎉 Unzipping Complete!")

# Run it
if os.path.exists(TARTAN_ROOT):
    unzip_tartanair()
else:
    print(f"❌ Path not found: {TARTAN_ROOT}")

🕵️ Scanning ../datasets/TartanAir for zip files...
📦 Found 4 zip files. Starting extraction...


100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 14.37it/s]

🎉 Unzipping Complete!


In [24]:
# === CELL 23: TARTANAIR PATH FINDER ===
import os
import glob

TARTAN_ROOT = '../datasets/TartanAir'

print(f"🕵️ Searching for ANY .png file in {TARTAN_ROOT}...")

# Recursive search for just ONE png file to see its path
# We limit to 1 match to be fast
pngs = glob.glob(os.path.join(TARTAN_ROOT, '**', '*.png'), recursive=True)

if len(pngs) > 0:
    print("✅ FOUND PNGs! Here is the structure:")
    example = pngs[0]
    print(f"   Full Path: {example}")
    
    # Analyze the path
    parts = example.split(os.sep)
    try:
        idx = parts.index('TartanAir')
        rel_path = "/".join(parts[idx:])
        print(f"   Relative:  {rel_path}")
    except:
        pass
        
    # Check if 'image_left' is duplicated in the path
    if example.count('image_left') > 1:
        print("   ⚠️  DETECTED DOUBLE FOLDER: 'image_left/image_left'")
    
else:
    print("❌ NO PNG FILES FOUND anywhere in TartanAir root.")
    print("   This means the data is definitely not extracted correctly.")
    
    # Check if P000/image_left folder exists and what's inside
    sample_dir = os.path.join(TARTAN_ROOT, 'office/Easy/P000/image_left')
    if os.path.exists(sample_dir):
        print(f"\n   Content of {sample_dir}:")
        print(os.listdir(sample_dir))
    else:
        print(f"\n   {sample_dir} does not exist.")

🕵️ Searching for ANY .png file in ../datasets/TartanAir...
✅ FOUND PNGs! Here is the structure:
   Full Path: ../datasets/TartanAir/neighborhood/Easy/P007/image_left/000067_left.png
   Relative:  TartanAir/neighborhood/Easy/P007/image_left/000067_left.png


In [27]:
# === CELL 23: THE FINAL TRAINING LOOP (COLLATE FIXED) ===
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
import time
import os
import re
import numpy as np

# --- 1. CONFIGURATION ---
BATCH_SIZE = 8       
NUM_WORKERS = 4      
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# --- 2. CUSTOM COLLATE FUNCTION (CRITICAL FIX) ---
def hexapod_collate(batch):
    """
    Custom collate to handle variable number of detection boxes.
    Images/Maps are stacked. Boxes are kept as a list.
    """
    collated = {}
    
    # Keys: left, right, teacher, disp, seg, det
    # 1. Stack Fixed-Size Data
    collated['left']    = torch.stack([item['left'] for item in batch])
    collated['right']   = torch.stack([item['right'] for item in batch])
    collated['teacher'] = torch.stack([item['teacher'] for item in batch])
    collated['disp']    = torch.stack([item['disp'] for item in batch])
    collated['seg']     = torch.stack([item['seg'] for item in batch])
    
    # 2. Keep Variable-Size Data as List
    collated['det']     = [item['det'] for item in batch]
    
    return collated

# --- 3. DATASET SETUP (WITH PFM FIX) ---
def read_pfm_fixed(file):
    with open(file, 'rb') as f:
        header = f.readline().decode().rstrip()
        color = (header == 'PF')
        dim_match = re.match(r'^(\d+)\s(\d+)\s$', f.readline().decode('utf-8'))
        width, height = map(int, dim_match.groups())
        scale = float(f.readline().decode().rstrip())
        
        if scale < 0:
            endian = '<'
            scale = -scale
        else:
            endian = '>'

        data = np.fromfile(f, endian + 'f')
        shape = (height, width, 3) if color else (height, width)
        data = np.reshape(data, shape)
        data = np.flipud(data)
        
    return data.copy() * scale

# Re-init Dataset
# (Ensure your RealFusedDataset class uses the updated read_pfm logic or patch it here)
import sys
sys.modules[__name__].read_pfm = read_pfm_fixed
real_ds = RealFusedDataset(REAL_ROOTS, mode='train')

# Split
train_size = int(0.95 * len(real_ds))
val_size = len(real_ds) - train_size
train_ds, val_ds = random_split(real_ds, [train_size, val_size])

# Loaders with COLLATE_FN
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                          collate_fn=hexapod_collate) # <--- ADDED THIS

val_loader =   DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, 
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=hexapod_collate) # <--- ADDED THIS

print(f"🚆 Data Ready: {len(train_ds)} Train, {len(val_ds)} Val samples.")

# --- 4. TRAINING SETUP ---
writer = SummaryWriter(log_dir="./logs")
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LEARNING_RATE, 
                                          steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS)

try:
    scaler = torch.amp.GradScaler('cuda')
except:
    scaler = torch.cuda.amp.GradScaler()

def save_checkpoint(epoch, loss):
    path = os.path.join(SAVE_DIR, f"hexapod_epoch_{epoch}.pth")
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': loss,
    }, path)
    print(f"💾 Checkpoint saved: {path}")

# --- 5. THE LOOP ---
def train_one_epoch(epoch_index):
    model.train()
    running_loss = 0.0
    valid_batches = 0
    
    from tqdm import tqdm
    pbar = tqdm(train_loader, desc=f"Epoch {epoch_index+1}/{NUM_EPOCHS}")
    
    for batch_idx, batch in enumerate(pbar):
        # Move Fixed Data to GPU
        l = batch['left'].to(DEVICE)
        r = batch['right'].to(DEVICE)
        t_img = batch['teacher'].to(DEVICE)
        
        # Targets
        det_targets = [t.to(DEVICE) for t in batch['det']]
        targets = {
            'disp': batch['disp'].to(DEVICE),
            'seg': batch['seg'].to(DEVICE),
            'det': det_targets
        }
        
        # 1. Teacher Pass
        with torch.no_grad():
            teacher_logits = teacher_seg(t_img)
            teacher_preds = {'seg': teacher_logits}
            
        # 2. Student Pass
        optimizer.zero_grad()
        
        try:
            autocast_ctx = torch.amp.autocast('cuda')
        except:
            autocast_ctx = torch.cuda.amp.autocast()
            
        with autocast_ctx:
            preds = model(l, r)
            loss, logs = criterion(preds, targets, teacher_preds)
            
        # --- NAN CHECK ---
        if torch.isnan(loss) or torch.isinf(loss):
            # print(f"⚠️ Warning: NaN Loss detected at batch {batch_idx}. Skipping.")
            # Zero out gradients just in case
            optimizer.zero_grad()
            continue
            
        # 3. Backward
        scaler.scale(loss).backward()
        
        # Unscale before clipping
        scaler.unscale_(optimizer)
        
        # Clip Gradients (Crucial for stability)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        # 4. Logging
        running_loss += loss.item()
        valid_batches += 1
        current_lr = scheduler.get_last_lr()[0]
        
        if batch_idx % 50 == 0:
            global_step = epoch_index * len(train_loader) + batch_idx
            writer.add_scalar('Loss/Total', loss.item(), global_step)
            for k, v in logs.items():
                writer.add_scalar(f'Loss/{k}', v, global_step)
            writer.flush() # <--- ADD THIS LINE    
        pbar.set_postfix({'loss': f"{loss.item():.2f}", 'lr': f"{current_lr:.2e}"})
        
    avg_loss = running_loss / (valid_batches + 1e-6)
    return avg_loss

# --- 6. EXECUTION ---
print("🚀 STARTING TRAINING...")
try:
    for epoch in range(NUM_EPOCHS):
        avg_loss = train_one_epoch(epoch)
        print(f"📉 Epoch {epoch+1} Finished. Avg Loss: {avg_loss:.4f}")
        save_checkpoint(epoch+1, avg_loss)
        
except KeyboardInterrupt:
    print("\n🛑 Training Interrupted by User. Saving Emergency Checkpoint...")
    save_checkpoint('INTERRUPTED', 0.0)

print("🏁 Training Complete.")

   Scanning FlyingThings3D...
   -> Found 21818 FT3D pairs.
   Scanning TartanAir...
   -> Found 38603 TartanAir samples.
   Scanning COCO 2017...
loading annotations into memory...
Done (t=9.80s)
creating index...
index created!
   -> Found 117266 COCO samples.
📊 Total Training Samples: 177687
🚆 Data Ready: 168802 Train, 8885 Val samples.
🚀 STARTING TRAINING...


Epoch 1/10:   1%|▎                           | 255/21100 [01:33<2:07:05,  2.73it/s, loss=2554.39, lr=4.00e-06]


ValueError: Caught ValueError in DataLoader worker process 3.
Original Traceback (most recent call last):
  File "/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
  File "/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 420, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
  File "/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 420, in <listcomp>
    return [self.dataset[self.indices[idx]] for idx in indices]
  File "/tmp/ipykernel_495/2899602516.py", line 100, in __getitem__
    transformed = self.transform_main(
 

In [ ]:
# === CELL 25: FIXED VISUALIZER ===
import matplotlib.pyplot as plt
import torch

def visualize_current_progress(model, dataset):
    model.eval()
    # Get a sample and ensure it has a batch dimension
    sample = dataset[0]
    l = sample['left'].unsqueeze(0).to(DEVICE)
    r = sample['right'].unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        preds = model(l, r)
        
        # Unpack the first item (Stereo)
        stereo_out = preds[0]
        
        # If it's a list (multi-scale), get the last one
        if isinstance(stereo_out, list):
            stereo_out = stereo_out[-1]
            
        # If it's still low res, upsample it
        if stereo_out.shape[-2:] != (480, 640):
            stereo_out = torch.nn.functional.interpolate(stereo_out, size=(480, 640), mode='bilinear')

    # Convert to Numpy
    # Shape is likely [1, 12, 480, 640] or [1, 1, 480, 640]
    disp_np = stereo_out.cpu().detach().numpy()[0] # Now [C, 480, 640]
    
    if disp_np.shape[0] > 1:
        # If we have 12 channels, the model hasn't learned to collapse them yet.
        # We take the mean across channels just to see the spatial structure.
        disp_img = disp_np.mean(axis=0)
        title_suffix = f"(Mean of {disp_np.shape[0]} channels)"
    else:
        disp_img = disp_np[0]
        title_suffix = ""

    # Plotting
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    # Input Image
    ax[0].imshow(sample['left'].squeeze().cpu().numpy(), cmap='gray')
    ax[0].set_title("Robot Input (Monochrome)")
    ax[0].axis('off')
    
    # Heatmap
    im = ax[1].imshow(disp_img, cmap='magma')
    ax[1].set_title(f"Model Prediction {title_suffix}")
    ax[1].axis('off')
    
    plt.colorbar(im, ax=ax[1], label="Activation Intensity")
    plt.tight_layout()
    plt.show()

visualize_current_progress(model, real_ds)